# Streamlit app: Legal RAG with recursive retrieval on a lexical graph

Upload a **PDF** and its **Reducto JSON**, ask a question, and the pipeline from the original notebook runs
(hybrid BM25 + vector search → pruning → full-clause expansion → link/footer hints → router/supervisor loop → answer).
The app also draws the **lexical graph** and the **definitions graph**, and can highlight the nodes/terms used for your last answer.

**How to use**
1. Run the cells top to bottom (the last cell prints a public `trycloudflare.com` URL).
2. Open the URL, paste your OpenAI key in the sidebar (or put it in Colab Secrets as `OPENAI_API_KEY`), upload both files, click **Build graphs & index**.
3. Ask questions in the *Ask a question* view; switch to the graph views to explore.

> Use a fresh OpenAI key stored in Colab Secrets. Never paste keys into notebook cells.

In [ ]:
%pip install -q "pymupdf>=1.24" "llama-index-core>=0.12,<0.15" "llama-index-embeddings-openai>=0.3" "llama-index-llms-openai>=0.3" "llama-index-retrievers-bm25>=0.5" "PyStemmer>=2.2" "openai>=1.60" "langgraph>=0.2" "langchain-core>=0.3" "tiktoken>=0.7" "networkx>=3.0" pandas numpy pyvis streamlit
print("done")

## (Optional) Load your OpenAI key from Colab Secrets
Skip this cell if you prefer typing the key into the app's sidebar.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("OPENAI_API_KEY loaded from Colab Secrets")
except Exception as e:
    print("No Colab secret found (you can enter the key in the app instead):", type(e).__name__)

## 1 · Pipeline module (`rag_pipeline.py`)
The notebook's functions and agents, unchanged apart from the small fixes listed in the notes at the end.

In [ ]:
%%writefile rag_pipeline.py
"""
RAG pipeline ported 1:1 from the notebook "Recursive Retrieval on a Lexical Graph (NetworkX)".

The notebook relied on module-level globals (LEXICAL_GRAPH, DEFINITIONS_GRAPH, local_nodes_map,
vector_index, ...). To keep the agent code identical, this module keeps those globals and fills
them in `build_pipeline(...)`. The Streamlit app (app.py) calls build_pipeline() once per upload and
run_query() per question.
"""
import csv
import hashlib
import html
import json
import math
import os
import pickle
from collections import Counter, defaultdict
from enum import Enum
from functools import lru_cache
from typing import Any, Callable, Dict, List, Optional, Set, Tuple, TypedDict

import fitz  # pymupdf
import networkx as nx
import numpy as np
import pandas as pd
import Stemmer
import tiktoken
from openai import OpenAI
from pydantic import BaseModel

from langchain_core.messages import BaseMessage  # noqa: F401  (kept from the notebook)
from langgraph.graph import END, START, StateGraph

from llama_index.core import (
    QueryBundle,
    Settings,
    SimpleKeywordTableIndex,
    StorageContext,
    VectorStoreIndex,
)
from llama_index.core.retrievers import BaseRetriever, KeywordTableSimpleRetriever, VectorIndexRetriever
from llama_index.core.schema import NodeWithScore, TextNode
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI as LLamaOpenAI
from llama_index.retrievers.bm25 import BM25Retriever

# --------------------------------------------------------------------------------------
# Global state (filled by build_pipeline)
# --------------------------------------------------------------------------------------
FILENAME: Optional[str] = None
EMBED_MODEL = "text-embedding-3-small"
MAX_EMBED_CHARS = 20_000  # stay safely below the 8191-token input limit
GRAPH_DIR = "./graph_data"
os.makedirs(GRAPH_DIR, exist_ok=True)

# Unstructured didn't need this, but Reducto needed a vertical stretch to get accurate results
HEIGHT_ADJUSTMENT_FACTOR = 0.96

openai_client: Optional[OpenAI] = None
LEXICAL_GRAPH: Optional[nx.MultiDiGraph] = None
DEFINITIONS_GRAPH: Optional[nx.DiGraph] = None
local_nodes_map: Dict[str, "MultiAgentSearchLocalNode"] = {}
vector_index = None
keyword_index = None
keywords_bm25_retriever = None
footer_nodes_by_page_number: Dict[int, List["MultiAgentSearchLocalNode"]] = defaultdict(list)

_log: Callable[[str], None] = print


def is_ready() -> bool:
    return LEXICAL_GRAPH is not None and DEFINITIONS_GRAPH is not None and len(local_nodes_map) > 0


# --------------------------------------------------------------------------------------
# Link detection (pymupdf + Reducto)
# --------------------------------------------------------------------------------------
import math  # noqa: E402,F811


def calculate_distance(point1: Tuple[float, float], point2: Tuple[float, float]) -> float:
    """Calculate the Euclidean distance between two points where a point is a two-item tuple for xy coords."""
    return math.sqrt((point1[0] - point2[0]) ** 2 + (point1[1] - point2[1]) ** 2)


def point_to_segment_distance(px: float, py: float, ax: float, ay: float, bx: float, by: float) -> float:
    """Calculate the minimum distance from point (px, py) to the segment (ax, ay) - (bx, by)."""
    dx, dy = bx - ax, by - ay
    if dx == dy == 0:  # the segment is a point
        return calculate_distance((px, py), (ax, ay))

    t = max(0, min(1, ((px - ax) * dx + (py - ay) * dy) / (dx * dx + dy * dy)))
    nearest = (ax + t * dx, ay + t * dy)
    return calculate_distance((px, py), nearest)


@lru_cache(maxsize=None)
def get_page_resolution_pymupdf(pdf_path: str, page_number: int) -> Tuple[float, float]:
    """Get the width and height of a specific page in a PDF document using pymupdf (cached: the PDF is opened once per page)."""
    with fitz.open(pdf_path) as pdf_document:
        # Ensure the page number is valid
        if page_number < 1 or page_number > pdf_document.page_count:
            raise ValueError("Page number is out of range.")
        # Load the specified page (page_number is 1-based index)
        page = pdf_document.load_page(page_number - 1)
        return page.rect.width, page.rect.height


def min_distance_to_bbox(point: Tuple[float, float], element_bbox_data: dict, pdf_path: str) -> float:
    """Calculate the minimum distance from a point to a 4-point bounding box."""
    # Normalise pymupdf point to pymupdf's page resolution
    pymupdf_width, pymupdf_height = get_page_resolution_pymupdf(pdf_path, element_bbox_data["page"])
    px, py = point
    px /= pymupdf_width
    py /= pymupdf_height

    # Normalise bbox points to Reducto's page resolution
    left, top, width, height = (
        element_bbox_data["left"],
        element_bbox_data["top"],
        element_bbox_data["width"],
        element_bbox_data["height"],
    )
    points = [
        (left, top),
        (left + width, top),
        (left + width, top + height),
        (left, top + height),
    ]
    bbox = [(x, y * HEIGHT_ADJUSTMENT_FACTOR) for x, y in points]

    distances = []
    # Calculate all combinations of point-to-segment distances
    for i in range(len(bbox)):
        ax, ay = bbox[i]
        bx, by = bbox[(i + 1) % len(bbox)]
        distances.append(point_to_segment_distance(px, py, ax, ay, bx, by))
    return min(distances)


def return_closest_elements(
    page_number: int,
    point: Tuple[float, float],
    elements_by_pages: defaultdict,
    pdf_path: str,
) -> list:
    """
    Given pymupdf point coordinates and a page number, return a list of Reducto elements on the same
    page sorted by distance to the point. Used to find the corresponding Reducto element to a PDF link
    destination.
    """
    elements = elements_by_pages[page_number]
    sorted_elements = sorted(
        elements,
        key=lambda element: min_distance_to_bbox(point, element["bbox"], pdf_path),
    )
    return sorted_elements


def get_elements_sort_by_pages(file_name: str) -> defaultdict:
    """Retrieve and organise elements by pages from a Reducto output JSON file."""
    elements_file = f"{file_name}.json"

    with open(elements_file, "r") as f:
        data = json.load(f)

    elements_by_pages = defaultdict(list)

    for chunk in data["result"]["chunks"]:
        for element in chunk["blocks"]:
            elements_by_pages[element["bbox"]["page"]].append(element)

    return elements_by_pages


def hash_element(element: dict) -> str:
    """Generate a hash for an element using its content and bounding box."""
    content = element["content"]
    bbox = str(element["bbox"]).encode("utf-8")
    return hashlib.md5(content.encode("utf-8") + bbox).hexdigest()


def unique_node_name(content, current_element_hash_id):
    """Generate a unique graph node ID from a content preview and the element hash."""
    clean_content = " ".join(content.split())
    node_name_content_preview = clean_content[:50] + "..." if len(clean_content) > 50 else clean_content
    # node IDs have to be unique in the graph, so add the element hash to the end to make it unique
    node_name_content_preview += "_"
    node_name_content_preview += current_element_hash_id
    return node_name_content_preview


def extract_links_and_sections_to_elements(
    pdf_path: str, elements_by_pages: defaultdict
) -> Tuple[pd.DataFrame, List[Tuple]]:
    """
    Extract links from a PDF document and associate them with Reducto elements based on proximity.

    Returns a DataFrame containing the link information and a list of tuples representing links
    between elements.
    """
    document = fitz.open(pdf_path)
    links_data = []
    links_tuples = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        links = page.get_links()

        for link in links:
            link_data = {}
            link_data["page_number"] = page_num + 1

            # Extract link text
            rect = fitz.Rect(link["from"])
            text = page.get_text("text", clip=rect)
            link_data["link_text"] = text.strip() if text else "No text"

            x, y, _, _ = link["from"]  # Get the coordinates of the link
            closest_elems = return_closest_elements(page_num + 1, (x, y), elements_by_pages, pdf_path)
            closest_elem = closest_elems[0] if closest_elems else None
            if closest_elem:
                link_data["guess_link_reducto_element"] = closest_elem["content"]
                link_element_id = hash_element(closest_elem)
            else:
                link_data["guess_link_reducto_element"] = "N/A"
                link_element_id = None

            # Determine link type and location
            if link["kind"] == fitz.LINK_GOTO:
                link_data["link_type"] = "internal"
                link_data["link_destination"] = "page " + str(link["page"] + 1 if link["page"] >= 0 else "Unknown")

                # Extract named destination if available
                dest = link.get("to", None)
                if dest:
                    link_data["named_destination"] = f"Coordinates: {dest}"
                    x, y = dest
                    target_page_num = link["page"] + 1
                    closest_elems = return_closest_elements(target_page_num, (x, y), elements_by_pages, pdf_path)
                    closest_elem = closest_elems[0] if closest_elems else None
                    if closest_elem:
                        link_data["guess_destination_reducto_element"] = closest_elem["content"]
                        link_destination_element_id = hash_element(closest_elem)
                    else:
                        link_data["guess_destination_reducto_element"] = "N/A"
                        link_destination_element_id = None
                else:
                    link_data["named_destination"] = "N/A"
                    link_data["guess_destination_reducto_element"] = "N/A"
                    link_destination_element_id = None

            elif link["kind"] == fitz.LINK_URI:
                link_data["link_type"] = "external"
                link_data["link_destination"] = link["uri"]
                link_data["named_destination"] = "N/A"
                link_data["guess_destination_reducto_element"] = "N/A"
                link_destination_element_id = link["uri"]
            else:
                link_data["link_type"] = "unknown"
                link_data["link_destination"] = "Unknown location"
                link_data["named_destination"] = "N/A"
                link_data["guess_destination_reducto_element"] = "N/A"
                link_destination_element_id = None

            links_tuples.append((link_element_id, link_destination_element_id))
            links_data.append(link_data)

    document.close()
    df = pd.DataFrame(links_data)

    # Replace NaN and infinite values
    df.replace([np.inf, -np.inf, np.nan], "N/A", inplace=True)

    return df, links_tuples


def save_to_csv(df: pd.DataFrame, output_path: str) -> None:
    """Save a DataFrame to a CSV file at the specified path."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_csv(output_path, index=False)


def analyse_links(filename: str, elements_by_pages: defaultdict) -> Tuple[pd.DataFrame, List[Tuple]]:
    """
    Analyse links in a PDF document, matching them to Reducto elements, and saving the results to a CSV file.

    Returns a DataFrame with the link data and a list of tuples representing links between elements.
    """
    pdf_path = f"{filename}.pdf"
    links_df, links_tuples = extract_links_and_sections_to_elements(pdf_path, elements_by_pages)
    save_to_csv(links_df, f"docs_links/{filename}-with-reducto.csv")
    return links_df, links_tuples


# --------------------------------------------------------------------------------------
# Configuration and helpers
# --------------------------------------------------------------------------------------
def configure_openai(api_key: str, embed_model: str) -> None:
    """Create the OpenAI client and point LlamaIndex at the same key + embedding model."""
    global openai_client, EMBED_MODEL
    EMBED_MODEL = embed_model
    os.environ["OPENAI_API_KEY"] = api_key
    openai_client = OpenAI(api_key=api_key)
    # EMBED_MODEL MUST match the model used to embed the node texts
    Settings.llm = LLamaOpenAI(model="gpt-4o", api_key=api_key)
    Settings.embed_model = OpenAIEmbedding(model_name=embed_model, api_key=api_key)


def save_graph(graph: nx.Graph, path: str) -> None:
    """Persist a graph (including node embeddings) to disk."""
    with open(path, "wb") as f:
        pickle.dump(graph, f)


def load_graph(path: str) -> nx.Graph:
    with open(path, "rb") as f:
        return pickle.load(f)


def embed_texts(texts: List[str], model: Optional[str] = None, batch_size: int = 100) -> List[List[float]]:
    """Embed a list of texts with OpenAI, in batches. Empty strings are replaced so the API doesn't reject them."""
    model = model or EMBED_MODEL
    embeddings: List[List[float]] = []
    for start in range(0, len(texts), batch_size):
        batch = [
            (t[:MAX_EMBED_CHARS] if t and t.strip() else "(empty content)")
            for t in texts[start : start + batch_size]
        ]
        response = openai_client.embeddings.create(model=model, input=batch)
        embeddings.extend(d.embedding for d in sorted(response.data, key=lambda d: d.index))
    return embeddings


# --------------------------------------------------------------------------------------
# Definitions graph
# --------------------------------------------------------------------------------------
class TermDefinition(BaseModel):
    """Pydantic schema used to describe the structured output of legal term-definition pairs for the GPT-4o model."""

    term: str
    definition: str


class TermDefinitionCSV(BaseModel):
    """Pydantic schema used to describe the structured output of legal term-definition pairs CSV for GPT-4o"""

    content: List[TermDefinition]


class DefinitionGraphBuilder:
    """
    Given a range of pages, the builder extracts legal terms and their definitions from a PDF
    (found at `file_path`), saves them to a CSV, and loads them into a NetworkX graph:

        Term --has_definition--> Definition
    """

    def __init__(
        self,
        definitions_page_ranges: List[Tuple[int, int]],  # page range starts count from 1
        file_path: str,
        definitions_csv_path: str,
        extract_with_llm: bool = True,
    ):
        self.definitions_page_ranges = definitions_page_ranges
        self.file_path = file_path
        self.definitions_csv_path = definitions_csv_path
        self.extract_with_llm = extract_with_llm
        self.graph: Optional[nx.DiGraph] = None

    def extract_and_save_to_csv(self) -> None:
        doc = fitz.open(self.file_path)

        all_terms_and_definitions = []

        # Flatten the list of ranges into individual pages
        page_numbers = []
        for start, end in self.definitions_page_ranges:
            page_numbers.extend(range(start, end + 1))

        for page_num in page_numbers:
            page = doc.load_page(page_num - 1)  # subtract 1 for 0-based index
            text = page.get_text("text")

            prompt = f"""
You will be shown a text from a page of a document containing legal terms and their definitions. Your task is to extract the legal terms and their corresponding definitions (verbatim) from the text and return them as the schema JSON.

Text:
{text}

Provide the extracted terms and definitions in the specified CSV format.
"""
            completion = openai_client.beta.chat.completions.parse(
                model="gpt-4o-2024-08-06",
                messages=[
                    {"role": "system", "content": "Extract the terms and definitions."},
                    {"role": "user", "content": prompt},
                ],
                response_format=TermDefinitionCSV,
            )

            terms_and_definitions = completion.choices[0].message.parsed.content
            _log(f"Definitions page {page_num}: {len(terms_and_definitions)} terms extracted")

            for item in terms_and_definitions:
                all_terms_and_definitions.append((item.term, item.definition))
        doc.close()

        os.makedirs(os.path.dirname(self.definitions_csv_path), exist_ok=True)
        with open(self.definitions_csv_path, "w", newline="") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(["Legal Term", "Definition"])
            for term, definition in all_terms_and_definitions:
                writer.writerow([term, definition])

    def load_pairs_from_csv(self) -> List[Tuple[str, str]]:
        with open(self.definitions_csv_path, "r", newline="") as csvfile:
            reader = csv.reader(csvfile)
            next(reader, None)  # skip header
            return [(row[0], row[1]) for row in reader if len(row) >= 2]

    def build_definitions_graph(self) -> nx.DiGraph:
        if self.extract_with_llm or not os.path.isfile(self.definitions_csv_path):
            self.extract_and_save_to_csv()

        graph = nx.DiGraph(name="Legal Definitions Graph")
        for term, definition in self.load_pairs_from_csv():
            definition_id = f"{term} (definition)"
            graph.add_node(term, label="Term", text=term)
            graph.add_node(definition_id, label="Definition", text=definition)
            graph.add_edge(term, definition_id, relation="has_definition")

        self.graph = graph
        _log(f"Built definitions graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
        return graph


# --------------------------------------------------------------------------------------
# Lexical graph
# --------------------------------------------------------------------------------------
class ElementType(str, Enum):  # subclass of str so JSON serializable
    PAGE = "Page"
    TITLE = "Title"
    FIGURE = "Figure"
    FOOTER = "Footer"
    LIST_ITEM = "List Item"
    HEADER = "Header"
    DOCUMENT = "Document"
    SECTION_HEADER = "Section Header"
    TEXT = "Text"
    TABLE = "Table"


ELEMENTS = [ElementType.value for ElementType in ElementType]

# Relation names, used as edge keys in the graph
CONTAINS = "contains"  # an element containing another element (e.g. a list item and its indented sub-items)
IS_PARENT = "is_parent"  # a section being the parent of its elements
LINKS_TO = "links_to"  # a link in the pdf, from one element to another
FOLLOWS = "follows"  # keeps track of section header flow in the document


class NetworkXLexicalGraphBuilder:
    """
    Builds the lexical graph of a document as a networkx.MultiDiGraph from the Reducto JSON + PDF links.

    Node ID  : "<first 50 chars of content>..._<element hash>"  (Document node: the file name, cover page: "Cover Page")
    Node data: label (element type), text, properties (Type/Hash/Node Name/Content/Page Number/Location), embedding
    Edge key : the relation name (contains / is_parent / follows / links_to)
    """

    def __init__(self, file_name: str):
        self.file_name = file_name

        json_path = f"{file_name}.json"
        _log("Loading Reducto JSON...")
        if not os.path.isfile(json_path):
            raise FileNotFoundError(f"Reducto JSON file not found: {json_path}")
        with open(json_path, "r") as f:
            self.json_data = json.load(f)
        if not self.json_data:
            raise ValueError("No data found in JSON file.")

        # (source element hash, destination element hash) pairs found by pymupdf + proximity matching
        _log("Detecting PDF links and matching them to Reducto elements...")
        elements_by_page_number = get_elements_sort_by_pages(file_name)
        self.links_tuples = analyse_links(file_name, elements_by_page_number)[1]

        self.graph = nx.MultiDiGraph(name=f"{file_name} Lexical Graph")
        # element hash -> node ID in the graph
        self.hash_to_node: Dict[str, str] = {}

    def _add_node(self, node_id: str, label: str, properties: Dict[str, Any], text: str) -> None:
        self.graph.add_node(node_id, label=label, properties=properties, text=text)

    def _link(self, head: Optional[str], tail: Optional[str], relation: str) -> bool:
        """Add head --relation--> tail. Silently ignores missing endpoints/self-loops. Re-adding the same relation is a no-op."""
        if head is None or tail is None or head == tail:
            return False
        if head not in self.graph or tail not in self.graph:
            return False
        self.graph.add_edge(head, tail, key=relation, relation=relation)
        return True

    def build_graph(self) -> nx.MultiDiGraph:
        doc_id = self.file_name
        cover_id = "Cover Page"

        self._add_node(doc_id, "Document", {"Type": "Document"}, text=doc_id)
        self._add_node(
            cover_id,
            "Page",
            {"Type": "Page", "Hash": "cover_page", "Node Name": cover_id, "Page Number": 1},
            text=cover_id,
        )
        self.hash_to_node["cover_page"] = cover_id
        self._link(doc_id, cover_id, CONTAINS)

        last_section_id: Optional[str] = None
        last_list_item_id: Optional[str] = None
        last_list_item_tabs = 0
        skipped = Counter()

        _log("Creating nodes and structural edges...")
        for reducto_chunk in self.json_data["result"]["chunks"]:
            for element in reducto_chunk["blocks"]:
                element_type = element["type"]
                if element_type not in ELEMENTS or element_type in ("Page", "Document"):
                    skipped[element_type] += 1  # e.g. "Page Number" blocks
                    continue

                page_number = int(element["bbox"]["page"])
                raw_content = element["content"]
                content = raw_content if raw_content.strip() else "(empty content)"
                element_hash = hash_element(element)
                node_id = unique_node_name(content, element_hash)

                self._add_node(
                    node_id,
                    element_type,
                    {
                        "Type": element_type,
                        "Hash": element_hash,
                        "Node Name": node_id,
                        "Content": content,
                        "Page Number": page_number,
                        "Location": str(element["bbox"]),
                    },
                    text=content,
                )
                self.hash_to_node[element_hash] = node_id

                # Everything on page 1 hangs off the cover page
                if page_number == 1:
                    self._link(cover_id, node_id, CONTAINS)
                    continue

                # No section heading above yet: attach to the document
                if last_section_id is None:
                    self._link(doc_id, node_id, CONTAINS)

                if element_type == "Section Header":
                    self._link(last_section_id, node_id, FOLLOWS)
                    last_section_id = node_id
                    last_list_item_id = None  # reset: start of a new section
                    self._link(doc_id, node_id, CONTAINS)

                elif element_type == "List Item":
                    current_tabs = raw_content.count("\t")
                    if last_list_item_id is not None and current_tabs > last_list_item_tabs:
                        # an indented list item is contained by the last top-level list item
                        self._link(last_list_item_id, node_id, CONTAINS)
                    else:
                        # first list item in the section, or a new top-level list item
                        last_list_item_id = node_id
                        last_list_item_tabs = current_tabs
                        self._link(last_section_id, node_id, IS_PARENT)

                else:
                    self._link(last_section_id, node_id, IS_PARENT)

        # Link elements to each other, using the pairs of element hashes found in the PDF's hyperlinks.
        # External (URL) links have no destination node, and links whose two ends resolve to the same element are useless, so both are skipped.
        _log("Adding links_to edges...")
        added_links = 0
        for source_hash, destination_hash in self.links_tuples:
            source = self.hash_to_node.get(source_hash)
            destination = self.hash_to_node.get(destination_hash)
            if self._link(source, destination, LINKS_TO):
                added_links += 1

        _log(
            f"Lexical graph: {self.graph.number_of_nodes()} nodes, {self.graph.number_of_edges()} edges "
            f"({added_links} links_to). Skipped block types: {dict(skipped)}"
        )
        return self.graph


def add_embeddings(graph: nx.Graph, model: Optional[str] = None) -> None:
    """Embed every node's text and store it as the node attribute 'embedding'."""
    model = model or EMBED_MODEL
    node_ids = list(graph.nodes)
    texts = [graph.nodes[n]["text"] for n in node_ids]
    _log(f"Embedding {len(texts)} nodes with {model}...")
    for node_id, embedding in zip(node_ids, embed_texts(texts, model)):
        graph.nodes[node_id]["embedding"] = embedding
    graph.graph["embed_model"] = model


# --------------------------------------------------------------------------------------
# LlamaIndex-compatible local node
# --------------------------------------------------------------------------------------
class MultiAgentSearchLocalNode(TextNode):
    """
    An extended version of LlamaIndex's TextNode, with support for Element Types from Reducto
    Built from a node of the NetworkX lexical graph (its text, embedding and properties).
    Supports a tree-recursive print function for outputting structure as YAML for LLM calls
    """

    # A TextNode class was used because it supports indexing with LlamaIndex.
    class Config:
        arbitrary_types_allowed = True
        json_encoders = {ElementType: lambda v: str(v)}  # Ensure ElementType is serialized as a string

    type: ElementType
    embedding: Optional[List[float]] = None
    metadata: Optional[Dict[str, Any]] = None
    children: Dict[str, "MultiAgentSearchLocalNode"] = {}
    context: Optional[Dict[str, Any]] = {}

    def __init__(self, *, graph: nx.Graph, graph_node_id: str, **data: Any):
        super().__init__(**data)
        try:
            attributes = graph.nodes[graph_node_id]
        except KeyError:
            raise ValueError(f"Node {graph_node_id} not found in graph")

        self.id_ = graph_node_id
        self.embedding = attributes.get("embedding")
        self.metadata = attributes["properties"]
        self.text = attributes["text"]

    # Implement hash and eq for use in dictionaries and sets
    def __hash__(self) -> int:
        return hash(self.node_id)

    def __eq__(self, other: object) -> bool:
        if isinstance(other, MultiAgentSearchLocalNode):
            return self.node_id == other.node_id
        return False

    # print node for prompt. use yaml to save tokens + better performance than json
    # recursive tree printing function
    # GPT will return with JSON
    def print_node_prompt(self, indent_level: int = 0, show_type=False) -> str:
        cleaned_text = " ".join(self.text.split())
        indent_unit = "  "
        indent = indent_unit * indent_level

        prompt = f"""{indent}- node_id: `{self.node_id}`"""

        if show_type:
            prompt += f"{indent}{indent_unit*2}type: {self.type}\n"

        prompt += f"{indent}{indent_unit*2}content: {cleaned_text}"

        if self.context.items():
            context = ""
            for key, value in self.context.items():
                if value not in [None, "", " "]:
                    context += f"{indent}{indent_unit*3}- {key}: {value}\n"
            prompt += f"{indent}{indent_unit*2}context:\n"
            prompt += f"{context}\n"
        children = list(self.children.values())
        if len(children) > 0:
            prompt += f"{indent}{indent_unit*2}children: \n"
            for child in children:
                prompt += f"{child.print_node_prompt(indent_level = indent_level + 5)}"
        return prompt


def graph_node_to_local_node(graph_node_id: str, graph: Optional[nx.Graph] = None) -> MultiAgentSearchLocalNode:
    """Build a local (LlamaIndex-compatible) node from a node ID in the lexical graph."""
    if graph is None:
        graph = LEXICAL_GRAPH
    return MultiAgentSearchLocalNode(
        graph=graph,
        graph_node_id=graph_node_id,
        type=ElementType(graph.nodes[graph_node_id]["label"]),
    )


# --------------------------------------------------------------------------------------
# Retrievers
# --------------------------------------------------------------------------------------
class BM25Keyword(BaseRetriever):
    """Custom retriever that performs both keyword search and BM25 search."""

    def __init__(self, mode: str = "OR", keyword_top_k=5, bm25_top_k=3) -> None:
        self._keyword_retriever = KeywordTableSimpleRetriever(
            index=keyword_index,
            max_keywords_per_query=keyword_top_k,
        )
        self._bm25_retriever = BM25Retriever.from_defaults(
            nodes=list(local_nodes_map.values()),
            similarity_top_k=bm25_top_k,
            stemmer=Stemmer.Stemmer("english"),
            language="english",
        )

        if mode not in ("AND", "OR"):
            raise ValueError("Invalid mode.")
        self._mode = mode
        super().__init__()

    def _retrieve(self, query_bundle: QueryBundle) -> List[MultiAgentSearchLocalNode]:
        """Retrieve nodes given query."""
        keyword_nodes = self._keyword_retriever.retrieve(query_bundle)
        bm25_nodes = self._bm25_retriever.retrieve(query_bundle)

        keyword_nodes_ids = {n.node.node_id for n in keyword_nodes}
        bm25_nodes_ids = {n.node.node_id for n in bm25_nodes}

        combined_dict = {n.node.node_id: n for n in keyword_nodes}
        combined_dict.update({n.node.node_id: n for n in bm25_nodes})

        if self._mode == "AND":
            retrieve_ids = keyword_nodes_ids.intersection(bm25_nodes_ids)
        else:
            retrieve_ids = keyword_nodes_ids.union(bm25_nodes_ids)

        return [combined_dict[rid] for rid in retrieve_ids]


class VectorBM25(BaseRetriever):
    """Custom retriever that performs both semantic search and BM25 search."""

    def __init__(self, mode: str = "OR", vector_top_k=2, bm25_top_k=6) -> None:
        self._vector_retriever = VectorIndexRetriever(index=vector_index, similarity_top_k=vector_top_k, verbose=True)
        self._bm25_retriever = BM25Retriever.from_defaults(
            nodes=list(local_nodes_map.values()),
            similarity_top_k=bm25_top_k,
            stemmer=Stemmer.Stemmer("english"),
            language="english",
        )
        self._vector_top_k = vector_top_k
        self._bm25_top_k = bm25_top_k
        if mode not in ("AND", "OR"):
            raise ValueError("Invalid mode.")
        self._mode = mode
        super().__init__()

    def _retrieve(self, query_bundle: QueryBundle) -> List[MultiAgentSearchLocalNode]:
        """Retrieve nodes given query."""
        vector_nodes = self._vector_retriever.retrieve(query_bundle)
        bm25_nodes = self._bm25_retriever.retrieve(query_bundle)

        vector_ids = {n.node.node_id for n in vector_nodes}
        keyword_ids = {n.node.node_id for n in bm25_nodes}

        combined_dict = {n.node.node_id: n for n in vector_nodes}
        combined_dict.update({n.node.node_id: n for n in bm25_nodes})

        if self._mode == "AND":
            retrieve_ids = vector_ids.intersection(keyword_ids)
        else:
            retrieve_ids = vector_ids.union(keyword_ids)

        return [combined_dict[rid] for rid in retrieve_ids]


def retrieve_with_keywords_bm25(query, verbose=False):
    retrieved_nodes: NodeWithScore = keywords_bm25_retriever.retrieve(QueryBundle(query_str=query))
    retrieved_nodes_local_form: MultiAgentSearchLocalNode = [local_nodes_map[node.node.node_id] for node in retrieved_nodes]
    if verbose:
        for node in retrieved_nodes:
            print(node)
    return retrieved_nodes_local_form


# --------------------------------------------------------------------------------------
# Recursive tree functions on subgraphs
# --------------------------------------------------------------------------------------
def prune_node_from_nodes(nodes: List[MultiAgentSearchLocalNode], node_id: str) -> None:
    """Prune a node tree by node_id."""

    def prune(node: MultiAgentSearchLocalNode) -> bool:
        if node.node_id == node_id:
            return True
        to_delete = []
        for child_id, child_node in node.children.items():
            if prune(child_node):
                to_delete.append(child_id)
        for child_id in to_delete:
            del node.children[child_id]
        return False

    nodes[:] = [root_node for root_node in nodes if not prune(root_node)]


def recursive_update(nodes: List[MultiAgentSearchLocalNode], node_id: str, property: str, property_value: Any) -> None:
    """Recursively search a node tree by node_id and update the specified property with the given value."""

    def update_node(node: MultiAgentSearchLocalNode) -> bool:
        if node.node_id == node_id:
            setattr(node, property, property_value)
            return True
        for child in node.children.values():
            if update_node(child):
                return True
        return False

    for root_node in nodes:
        if update_node(root_node):
            break


def recursive_add_context(nodes: List[MultiAgentSearchLocalNode], node_id: str, context_key: str, context_value: str) -> None:
    """Recursively search a node tree by node_id and add a key-value pair to the context dictionary of the matching node."""

    def add_context_to_node(node: MultiAgentSearchLocalNode) -> bool:
        if node.node_id == node_id:
            node.context[context_key] = context_value
            return True
        for child in node.children.values():
            if add_context_to_node(child):
                return True
        return False

    for root_node in nodes:
        if add_context_to_node(root_node):
            break

    return nodes


def recursive_append_context_items(
    nodes: List[MultiAgentSearchLocalNode], node_id: str, context_key: str, new_items: List[str]
) -> None:
    """Recursively search a node tree by node_id and add a key-value pair to the context dictionary of the matching node."""

    def add_context_link_to_node(node: MultiAgentSearchLocalNode) -> bool:
        if node.node_id == node_id:
            node_links = node.context.get(context_key, [])
            node_links.append(new_items)
            node.context[context_key] = node_links
            return True
        for child in node.children.values():
            if add_context_link_to_node(child):
                return True
        return False

    for root_node in nodes:
        if add_context_link_to_node(root_node):
            break

    return nodes


def recursive_check_if_exists(nodes: List[MultiAgentSearchLocalNode], node_id: str) -> bool:
    """Recursively search a node tree by node_id and check if the node exists."""

    def check_if_exists(node: MultiAgentSearchLocalNode) -> bool:
        if node.node_id == node_id:
            return True
        for child in node.children.values():
            if check_if_exists(child):
                return True
        return False

    for root_node in nodes:
        if check_if_exists(root_node):
            return True

    return False


# --------------------------------------------------------------------------------------
# Tools for agents
# --------------------------------------------------------------------------------------
class PruneNodeIdResponse(BaseModel):
    """Pydantic schema used to describe a node via node_id being returned by the LLM."""

    node_id: str
    reason: str


class PruneNodeIdsResponse(BaseModel):
    """Pydantic schema used to describe nodes via node_ids being returned by the LLM."""

    content: List[PruneNodeIdResponse]


def prune_nodes(query: str, current_nodes: List[MultiAgentSearchLocalNode]) -> List[MultiAgentSearchLocalNode]:
    """LLM call to remove any nodes that don't contain relevant information to the query."""

    starter_prompt = f"""
    You are an intelligent agent responsible for post-retrieval filtration of retrieved nodes from a document. The nodes are to answer the query:
    ```{query}```

    Below is a list of nodes that were automatically retrieved. Your task is to identify the nodes that do not contain relevant information to the query, and a short reason as to why they should be pruned.

    Be warned that deleting nodes will delete any sub-nodes listed within them.

    Nodes are identified by node_id. Return a list of (node_id, reason) where reason is a short explanation as to why the node should be pruned. node_ids must be quoted in backticks.

    """

    all_flattened_nodes: Dict[str, MultiAgentSearchLocalNode] = {}

    def recurse_collect_nodes(current_nodes: List[MultiAgentSearchLocalNode]):
        """Recursively iterate via DFS on nodes, to index them by page number."""
        for node in current_nodes:
            all_flattened_nodes[node.id_] = node

            if len(node.children) > 0:
                recurse_collect_nodes(list(node.children.values()))

    recurse_collect_nodes(current_nodes)

    printout = ""
    for node in all_flattened_nodes.values():
        printout += node.print_node_prompt()

    completion = openai_client.beta.chat.completions.parse(
        model="gpt-4o-2024-08-06",
        messages=[
            {"role": "system", "content": starter_prompt},
            {"role": "user", "content": printout},
        ],
        response_format=PruneNodeIdsResponse,
    )

    node_id_response = completion.choices[0].message.parsed.content

    prune_dictionary_results: Dict[str, str] = {}

    if isinstance(node_id_response, list):
        for item in node_id_response:
            if isinstance(item, PruneNodeIdResponse):
                if item.node_id[0] == "`":
                    node_id = item.node_id[1:-1]
                else:
                    node_id = item.node_id
                prune_dictionary_results[node_id] = item.reason
                prune_node_from_nodes(current_nodes, node_id)

    return current_nodes, prune_dictionary_results


def fetch_whole_clause_lists(
    current_nodes: List[MultiAgentSearchLocalNode], graph: Optional[nx.MultiDiGraph] = None
) -> List[MultiAgentSearchLocalNode]:
    """
    Given a list of nodes will retrieve and replace subclauses with full clauses.
    Assumes that we have a flat sequence of nodes, not a tree of nodes.
    """
    if graph is None:
        graph = LEXICAL_GRAPH

    def is_list_item(node_id: str) -> bool:
        return graph.nodes[node_id]["label"] == ElementType.LIST_ITEM.value

    def structural_parents(node_id: str) -> List[str]:
        # whoever "contains" this node or is its "is_parent"
        return [head for head, _, rel in graph.in_edges(node_id, keys=True) if rel in (CONTAINS, IS_PARENT)]

    def list_item_children(node_id: str) -> List[str]:
        # edges are stored in insertion order, which is document order
        return [
            tail for _, tail, rel in graph.out_edges(node_id, keys=True) if rel == CONTAINS and is_list_item(tail)
        ]

    # clause title id -> clause title node (with all of its subclauses attached as children)
    clause_titles: Dict[str, MultiAgentSearchLocalNode] = {}
    # subclause id -> id of the clause title it belongs to
    subclause_to_clause_map: Dict[str, str] = {}

    def register_clause(title_id: str) -> None:
        if title_id in clause_titles:
            return
        title_node = graph_node_to_local_node(title_id, graph)
        for subclause_id in list_item_children(title_id):
            title_node.children[subclause_id] = graph_node_to_local_node(subclause_id, graph)
            subclause_to_clause_map[subclause_id] = title_id
        clause_titles[title_id] = title_node

    # Step 1: find the clause title of every List Item we were given.
    # Step 2 (inside register_clause): fetch all subclauses of each clause title.
    for node in current_nodes:
        if node.type != ElementType.LIST_ITEM:
            continue
        list_item_parents = [p for p in structural_parents(node.id_) if is_list_item(p)]
        register_clause(list_item_parents[0] if list_item_parents else node.id_)

    final_nodes = []
    seen_clauses: Set[str] = set()
    for current_node in current_nodes:
        # if not a list item, keep as is
        if current_node.type != ElementType.LIST_ITEM:
            final_nodes.append(current_node)
            continue

        # a clause title, or a subclause (look up its clause title)
        if current_node.id_ in clause_titles:
            clause_id = current_node.id_
        else:
            clause_id = subclause_to_clause_map.get(current_node.id_)

        # only add each full clause once: later subclauses of a clause we've already added are dropped
        if clause_id is not None and clause_id not in seen_clauses:
            seen_clauses.add(clause_id)
            final_nodes.append(clause_titles[clause_id])

    return final_nodes


class RelevantTerms(BaseModel):
    """Pydantic schema used to describe the defined terms the LLM picked out of a query."""

    terms: List[str]


def definitions_search(query: str, graph: Optional[nx.DiGraph] = None) -> Dict[str, str]:
    """
    Search for definitions of terms in a question prompt and return them as a dictionary.

    The LLM only chooses WHICH defined terms appear in the query; the definitions themselves
    are then read verbatim from the graph, so they can't be hallucinated.
    """
    if graph is None:
        graph = DEFINITIONS_GRAPH

    # all defined terms in the graph, matched case-insensitively
    term_lookup = {node.lower(): node for node, data in graph.nodes(data=True) if data["label"] == "Term"}
    if not term_lookup:  # no definitions section -> nothing to look up (and no LLM call needed)
        return {}

    prompt = f"""
You are given a list of legal terms that have definitions, and a user query.
Return only the terms from the list that are relevant to, or appear in, the query. Use the exact spelling from the list.

Defined terms:
{json.dumps(list(term_lookup.values()))}
"""
    completion = openai_client.beta.chat.completions.parse(
        model="gpt-4o-2024-08-06",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": query},
        ],
        response_format=RelevantTerms,
    )

    definitions_dict: Dict[str, str] = {}
    for term in completion.choices[0].message.parsed.terms:
        graph_term = term_lookup.get(term.strip().lower())
        if graph_term is None:
            continue
        for definition_id in graph.successors(graph_term):
            if graph.nodes[definition_id]["label"] == "Definition":
                definitions_dict[graph_term] = graph.nodes[definition_id]["text"]
                break

    return definitions_dict


def print_prompt_definitions_dict(definitions_dict):
    prompt = "Relevant Definitions:\n"
    for term, definition in definitions_dict.items():
        prompt += f"{term}: {definition}\n"
    return prompt


# --------------------------------------------------------------------------------------
# Node context: footers + links
# --------------------------------------------------------------------------------------
class NodeIdFooterResponse(BaseModel):
    """Pydantic schema used to describe the structured output of node-id to footer info pair"""

    node_id: str
    footer_text: str
    further_explore_footer: bool


class NodeIdFooterResponses(BaseModel):
    """Pydantic schema used to describe the structured output of node-id to footer info pairs."""

    content: List[NodeIdFooterResponse]


def fill_nodes_with_footer_context(
    query: str, current_nodes: List[MultiAgentSearchLocalNode]
) -> Tuple[List[MultiAgentSearchLocalNode], int]:
    """Takes nodes and returns a list of footer contexts to fill them with."""

    starter_prompt = f"""
    You are an agent responsible for the structured extraction of a document page to answer the question:
    ```{query}```
    Below are two lists of OCR'd text, one for nodes and one for footers. All the text was taken from one page. Some of these nodes make references to the footers below, possibly through footnote numbers.

    Your task is, for each node_id, to identify if it contains (1) any footer information, and (2) if the footer requires further reading ON THE SAME DOCUMENT. Return only node_ids that contain footer info.

    For each node:
    1. deduce from the context and footnode numbering if any footer content might be relevant to the node given the query. If so, extract that footer content (it can be a partial or whole extract) and assign it to the node. If there is no footer info, SKIP the node. Only return nodes that have footer content.

    2. Some of the footer content may require the reader to consult other parts of the document (it must be an internal link pointing to other paragraphs/sections, not external reading). If so, set the further_explore_footer flag to true, else false.

    Then return a list of (node_id, relevant_footer_content, further_explore_footer). node_ids must be quoted in backticks.

    """

    nodes_by_page_number: Dict[int, List[MultiAgentSearchLocalNode]] = defaultdict(list)

    # collection of which nodes to mark with footer context: node_id -> (footer_text, further_explore_footer)
    node_id_footer_context: Dict[str, Tuple[str, bool]] = {}

    def recurse_index_nodes(current_nodes: List[MultiAgentSearchLocalNode]):
        """Recursively iterate via DFS on nodes, to index them by page number."""
        for node in current_nodes:
            page_number = int(node.metadata["Page Number"])
            nodes_by_page_number[page_number].append(node)

            if len(node.children) > 0:
                recurse_index_nodes(list(node.children.values()))

    recurse_index_nodes(current_nodes)

    for page_number, nodes in nodes_by_page_number.items():
        printout_of_curent_nodes = f"Nodes on page {page_number}"
        for node in nodes:
            printout_of_curent_nodes += node.print_node_prompt()

        printout_of_footers = f"Footers on page {page_number}:"
        for footer in footer_nodes_by_page_number[page_number]:
            printout_of_footers += footer.print_node_prompt()

        completion = openai_client.beta.chat.completions.parse(
            model="gpt-4o-2024-08-06",
            messages=[
                {"role": "system", "content": starter_prompt},
                {"role": "user", "content": printout_of_curent_nodes},
                {"role": "user", "content": printout_of_footers},
            ],
            response_format=NodeIdFooterResponses,
        )

        node_id_footer_response = completion.choices[0].message.parsed.content

        if isinstance(node_id_footer_response, list):
            for item in node_id_footer_response:
                if isinstance(item, NodeIdFooterResponse):
                    # sometimes GPT returns node ids with or without backticks
                    if item.node_id[0] == "`":
                        node_id = item.node_id[1:-1]
                    else:
                        node_id = item.node_id
                    node_id_footer_context[node_id] = (item.footer_text, item.further_explore_footer)

    change_count = 0
    # Based on the dict of node_id - footer context
    for node_id, (footer_text, further_explore_footer) in node_id_footer_context.items():
        if footer_text not in ["", " ", None]:
            recursive_add_context(current_nodes, node_id, "footer info", footer_text)
            recursive_add_context(current_nodes, node_id, "further explore footer?", str(further_explore_footer))
            change_count += 1

    return current_nodes, change_count


def fill_nodes_with_link_hints(
    current_nodes: List[MultiAgentSearchLocalNode], graph: Optional[nx.MultiDiGraph] = None
) -> Tuple[List[MultiAgentSearchLocalNode], int]:
    """Given a tree/list of nodes, update the nodes with hints to the LLM if any of the nodes contain links."""
    if graph is None:
        graph = LEXICAL_GRAPH

    # some nodes are nested, so recurse to get all nodes
    flattened_nodes: Dict[str, MultiAgentSearchLocalNode] = {}

    link_dict: Dict[str, List[str]] = defaultdict(list)

    def recurse_index_nodes(current_nodes: List[MultiAgentSearchLocalNode]):
        for node in current_nodes:
            flattened_nodes[node.id_] = node

            if len(node.children) > 0:
                recurse_index_nodes(list(node.children.values()))

    recurse_index_nodes(current_nodes)

    # head links_to tail: if the head is in our flattened nodes, then it links to tail
    for node_id in flattened_nodes:
        if node_id not in graph:
            continue
        destinations = [tail for _, tail, rel in graph.out_edges(node_id, keys=True) if rel == LINKS_TO]
        if destinations:
            link_dict[node_id] = list(dict.fromkeys(destinations))  # de-duplicate, keep order

    # add context markers inside any node with a link
    change_count = 0
    for link_node_id, destinations in link_dict.items():
        recursive_append_context_items(current_nodes, link_node_id, "link", str(destinations))
        change_count += 1

    return current_nodes, change_count


# --------------------------------------------------------------------------------------
# LangGraph multi-agent workflow
# --------------------------------------------------------------------------------------
TOKEN_LIMIT = 32000
_tokenizer = None


def count_tokens(text: str) -> int:
    global _tokenizer
    if _tokenizer is None:
        _tokenizer = tiktoken.get_encoding("o200k_base")
    return len(_tokenizer.encode(text))


def count_tokens_for_nodes(nodes: List[MultiAgentSearchLocalNode]) -> int:
    return sum(count_tokens(node.print_node_prompt()) for node in nodes)


debug_markdown = "# Multi-Agent Log\n"
debug_markdown += "## Pass 1\n"


class AgentState(TypedDict):
    query: str
    definitions: Dict[str, str]
    previous_nodes: List[MultiAgentSearchLocalNode]
    last_fetched_context_nodes: List[MultiAgentSearchLocalNode]
    node_links_to_fetch: List[str]
    node_footers_to_fetch: Dict[str, str]
    allow_continue: bool
    search_failures: List[str]
    markdown_debug: str
    pass_count: int
    answer: str  # (added for the app) the final answer text


def inital_search_agent(state: AgentState) -> AgentState:
    state["markdown_debug"] += f"### Initial Search Agent\n"

    vector_bm25_retriever = VectorBM25(vector_top_k=1, bm25_top_k=7, mode="OR")

    def retrieve_with_vector_bm25(query, verbose=False):
        retrieved_nodes: NodeWithScore = vector_bm25_retriever.retrieve(QueryBundle(query_str=query))
        retrieved_nodes_local_form: MultiAgentSearchLocalNode = [
            local_nodes_map[node.node.node_id] for node in retrieved_nodes
        ]
        if verbose:
            for node in retrieved_nodes:
                print(node)
        return retrieved_nodes_local_form

    # fetch nodes by vector and bm25 search
    initial_retrieved_nodes = retrieve_with_vector_bm25(state["query"])

    # LLM prunes nodes that are not relevant
    initial_pruned_nodes, _ = prune_nodes(state["query"], initial_retrieved_nodes)

    state["markdown_debug"] += (
        f" - retrieved {len(initial_retrieved_nodes)} nodes, pruned to {len(initial_pruned_nodes)} nodes\n"
    )

    state["last_fetched_context_nodes"] = initial_pruned_nodes
    return state


def definition_agent(state: AgentState) -> AgentState:
    # fetch definitions
    state["markdown_debug"] += f"### Definitions Agent\n"
    definitions_dict = definitions_search(state["query"])
    state["definitions"] = definitions_dict

    state["markdown_debug"] += f"Retrieved {len(definitions_dict)} definitions for: \n"
    for term, definition in definitions_dict.items():
        state["markdown_debug"] += f"  - **{term}** : {definition}\n"
    return state


def context_fetch_tool(state: AgentState) -> AgentState:
    state["markdown_debug"] += f"### Context Fetched for nodes:\n"
    current_nodes = state["last_fetched_context_nodes"]

    # expand search to get full clauses
    full_clause_nodes = fetch_whole_clause_lists(current_nodes)

    # mark links with nodes
    full_clause_nodes_with_link_hints, change_count = fill_nodes_with_link_hints(full_clause_nodes)

    state["last_fetched_context_nodes"] = full_clause_nodes_with_link_hints
    state["markdown_debug"] += (
        f"- fetched full clauses, added to {len(full_clause_nodes_with_link_hints)} nodes. {change_count} new links\n"
    )

    return state


def footer_parsing_agent(state: AgentState) -> AgentState:
    # fill nodes with footer context
    try:
        current_nodes, changes_count = fill_nodes_with_footer_context(state["query"], state["last_fetched_context_nodes"])

        state["last_fetched_context_nodes"] = current_nodes
        state["markdown_debug"] += f"- filled nodes with footer context\n"

        state["markdown_debug"] += f"\n```\n"
        for node in current_nodes:
            state["markdown_debug"] += node.print_node_prompt()

        state["markdown_debug"] += f"\n```\n"

    except Exception:
        state["search_failures"].append("footer parsing failed.")
        state["markdown_debug"] += f"- footer parsing failed\n"

    return state


def supervisor_agent(state: AgentState) -> AgentState:
    state["markdown_debug"] += f"### Supervisor Agent\n"
    # Look for search failures. This might be an instance where multiple searches were made for certain parts of the document, but no relevant information was found.
    # This means that the search has to be ended prematurely to prevent infinite loops.
    printout = ""
    for node in state["previous_nodes"]:
        printout += node.print_node_prompt()
    for node in state["last_fetched_context_nodes"]:
        printout += node.print_node_prompt()

    prompt = f"""
You are a supervisor agent overseeing the multi-agent retrieval process of graph nodes from a document. The nodes are to answer the query:
```{state['query']}```


Below is a list of nodes that were automatically retrieved, followed by a list of errors. If there are many similar, repeated errors in the retrieval process , where no further linked or relevant nodes could be retrieved, return END to end the process. Else return CONTINUE.
Return only a single word, either END or CONTINUE.
"""

    completion = openai_client.beta.chat.completions.parse(
        model="gpt-4o-2024-08-06",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": printout},
            {"role": "user", "content": str(state["search_failures"])},
        ],
    )

    response: str = completion.choices[0].message.content

    if "end" in response.lower():
        state["allow_continue"] = False
        return state
    state["markdown_debug"] += f"- **supervisor**: continue\n"

    # check if requests will be in token limit
    # chatgpt is 32k, use 32k as baseline
    tokens = count_tokens_for_nodes(state["previous_nodes"]) + count_tokens_for_nodes(state["last_fetched_context_nodes"])
    if tokens > TOKEN_LIMIT:
        state["markdown_debug"] += f"- warning: tokens over recommended limit\n"
        previous_nodes, changes_1 = prune_nodes(state["query"], state["previous_nodes"])
        last_fetched_context_nodes, changes_2 = prune_nodes(state["query"], state["last_fetched_context_nodes"])
        # (the notebook called count_tokens() on a list here, which would crash; count_tokens_for_nodes is what was meant)
        new_tokens = count_tokens_for_nodes(state["previous_nodes"]) + count_tokens_for_nodes(
            state["last_fetched_context_nodes"]
        )

        if new_tokens > TOKEN_LIMIT:
            state["markdown_debug"] += f"- Warning: Tried pruning, but still too many tokens\n"
        if new_tokens < tokens:
            state["previous_nodes"] = previous_nodes
            state["last_fetched_context_nodes"] = last_fetched_context_nodes

            for node_id, reason in changes_1.items():
                state["markdown_debug"] += f"- pruned node {node_id} due to {reason}\n"

            for node_id, reason in changes_2.items():
                state["markdown_debug"] += f"- pruned node {node_id} due to {reason}\n"

    return state


class FooterSearchResponse(BaseModel):
    """Pydantic schema used to contain the response for a node that needs a footer search, and the query involved."""

    node_id: str
    search_query: str


class RouterAgentFooterResponse(BaseModel):
    """Pydantic schema used to describe the structured output of the router agent."""

    node_ids_for_footer_search: List[FooterSearchResponse]


class RouterAgentLinkResponse(BaseModel):
    """Pydantic schema used to describe the structured output of the router agent."""

    node_ids_for_link_retrieval: List[str]


def router_agent(state: AgentState) -> AgentState:
    # decide if process should should stop or continue
    state["markdown_debug"] += f"### Router Agent\n"

    starter_prompt_footer = f"""
        You are an intelligent agent overseeing a multi-agent retrieval process of graph nodes from a document. These nodes are to answer the query:
        ```{state['query']}```

        Below this request is a list of nodes that were automatically retrieved.

        You must determine if the list of nodes is enough to answer the query. If there isn't enough information, you must identify any relevant footer information in the nodes.

        A node can footer information asking to look in another section/part of the document, which will require a separate natural language search.
        Example: If the footer says "see paragraph x", a search query e.g. "Return paragraph x to answer the query '{state['query']}'" should be made.

        If there are no further nodes worth analysing, return an empty response. ONLY RETURN QUERIES FOR FOOTERS THAT ARE RELEVANT TO ANSWERING THE QUERY

        Else, if any relevant nodes require a footer search, specify the node_id and the search query.
        Nodes are identified by node_id and must be quoted in backticks.


    """

    starter_prompt_link = f"""
        You are an intelligent agent overseeing a multi-agent retrieval process of graph nodes from a document. These nodes are to answer the query:
        ```{state['query']}```

        Below this request is a list of nodes that were automatically retrieved.

        You must determine if the list of nodes is enough to answer the query. If there isn't enough information, you must identify any linked nodes that could be worth exploring.

        If there are no further nodes worth analysing, return an empty response.

        Return a list of node_ids. ONLY RETURN NODE_IDS for NODES THAT ARE RELEVANT TO ANSWERING THE QUERY. Nodes are identified by node_id and must be quoted in backticks.
    """

    # collect latest nodes, and all nodes
    last_fetched_nodes_flattened: Dict[str, MultiAgentSearchLocalNode] = {}
    all_nodes_flattened: Dict[str, MultiAgentSearchLocalNode] = {}

    def recurse_collect_last_fetched_nodes(current_nodes: List[MultiAgentSearchLocalNode]):
        """Recursively iterate via DFS on nodes, to index them by page number."""
        for node in current_nodes:
            last_fetched_nodes_flattened[node.id_] = node
            all_nodes_flattened[node.id_] = node

            if len(node.children) > 0:
                recurse_collect_last_fetched_nodes(list(node.children.values()))

    recurse_collect_last_fetched_nodes(state["last_fetched_context_nodes"])

    def recurse_collect_all_nodes(current_nodes: List[MultiAgentSearchLocalNode]):
        """Recursively iterate via DFS on nodes, to index them by page number."""
        for node in current_nodes:
            all_nodes_flattened[node.id_] = node

            if len(node.children) > 0:
                recurse_collect_all_nodes(list(node.children.values()))

    recurse_collect_all_nodes(state["last_fetched_context_nodes"])

    printout = ""
    for node in last_fetched_nodes_flattened.values():
        printout += node.print_node_prompt()

    completion = openai_client.beta.chat.completions.parse(
        model="gpt-4o-2024-08-06",
        messages=[
            {"role": "system", "content": starter_prompt_link},
            {"role": "user", "content": printout},
        ],
        response_format=RouterAgentLinkResponse,
    )

    # Parse the response content
    explore_response = json.loads(completion.choices[0].message.content)

    # Extract node_ids_for_link_retrieval
    # Ensure that the node_ids are not already in the all_nodes_flattened
    node_ids_for_link_retrieval = []
    for node_id in explore_response["node_ids_for_link_retrieval"]:
        # sometimes GPT returns node ids with or without backticks
        if node_id[0] == "`":
            node_id = node_id[1:-1]
        if all_nodes_flattened.get(node_id) is None:
            node_ids_for_link_retrieval.append(node_id)

    completion = openai_client.beta.chat.completions.parse(
        model="gpt-4o-2024-08-06",
        messages=[
            {"role": "system", "content": starter_prompt_footer},
            {"role": "user", "content": printout},
        ],
        response_format=RouterAgentFooterResponse,
    )

    # Parse the response content
    explore_response = json.loads(completion.choices[0].message.content)

    # Initialize the dictionary for node_ids_for_footer_search
    node_ids_for_footer_search = {}

    # Populate the dictionary with node_id and search_query pairs
    for footer_response in explore_response["node_ids_for_footer_search"]:
        node_id = footer_response["node_id"]
        # sometimes GPT returns node ids with or without backticks
        if node_id[0] == "`":
            node_id = node_id[1:-1]

        node_ids_for_footer_search[node_id] = footer_response["search_query"]

    # Update the state with the node_ids_for_link_retrieval and node_ids_for_footer_search
    state["node_links_to_fetch"] = node_ids_for_link_retrieval
    state["node_footers_to_fetch"] = node_ids_for_footer_search

    # debug prints
    state["markdown_debug"] += f"- found {len(node_ids_for_link_retrieval)} nodes to fetch links for\n"
    for node_id in node_ids_for_link_retrieval:
        state["markdown_debug"] += f"    - {node_id}\n"
    state["markdown_debug"] += f" - found {len(node_ids_for_footer_search)} nodes to fetch footers for\n"
    for node_id, search_query in node_ids_for_footer_search.items():
        state["markdown_debug"] += f"    - `{node_id}` \n"
        state["markdown_debug"] += f"      - **Search query:** {search_query}\n"

    return state


def answering_agent(state: AgentState) -> AgentState:
    state["markdown_debug"] += f"### Answering Agent\n"
    # answer the query
    prompt = f"""
You are an answering agent. You will be given a list of document nodes that were automatically retrieved by the system. These nodes are to answer the query:
```{state['query']}```

Give references to sections/paragraphs if possible, but do not output full node ids with backticks and the hash.
    """

    printout = ""
    for node in state["previous_nodes"]:
        printout += node.print_node_prompt()

    for node in state["last_fetched_context_nodes"]:
        printout += node.print_node_prompt()

    completion = openai_client.chat.completions.create(
        model="gpt-4o-2024-08-06",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": printout},
        ],
    )

    message = completion.choices[0].message.content

    state["answer"] = message
    state["markdown_debug"] += f"## Answer to Query: \n{message}\n"
    state["markdown_debug"] += f"## Source Nodes: \n```\n{printout}\n```"

    return state


def recursive_retrieval(state: AgentState) -> AgentState:
    state["markdown_debug"] += f"### Recursive Retrieval\n"

    current_nodes = state["last_fetched_context_nodes"]

    for current_node in current_nodes:
        state["previous_nodes"].append(current_node)

    new_current_nodes = []

    # look up the nodes to fetch by id
    state["markdown_debug"] += f"{len(state['node_links_to_fetch'])} nodes with link(s)\n"

    for node_id in state["node_links_to_fetch"]:
        # sometimes GPT returns node ids with or without backticks
        if node_id[0] == "`":
            node_id = node_id[1:-1]
        if node_id in local_nodes_map:
            new_current_nodes.append(local_nodes_map[node_id])
            state["markdown_debug"] += f"  - {node_id}\n"
        else:
            state["search_failures"].append(f"Failed to fetch node with id: {node_id}")

    state["markdown_debug"] += f"{len(state['node_footers_to_fetch'])} nodes with footer info\n"

    for node_id, search_query in state["node_footers_to_fetch"].items():
        # fetch nodes by keyword and bm25 search
        footer_retrieved_nodes = retrieve_with_keywords_bm25(search_query)
        # LLM prunes nodes that are not relevant
        footer_retrieved_nodes, _ = prune_nodes(search_query, footer_retrieved_nodes)
        state["markdown_debug"] += f" - {node_id}\n"
        state["markdown_debug"] += f"  - **Search query:** {search_query}\n"
        state["markdown_debug"] += f"  - **Returned:** \n"

        for node in footer_retrieved_nodes:
            state["markdown_debug"] += f"    - {node.id_}\n"
            new_current_nodes.append(node)

        # if no nodes fetched, log failure
        if len(footer_retrieved_nodes) == 0:
            state["search_failures"].append(f"Failed to fetch nodes for query: {search_query}")

    state["last_fetched_context_nodes"] = new_current_nodes
    state["pass_count"] += 1
    state["markdown_debug"] += f"# PASS {state['pass_count']}\n"
    state["node_footers_to_fetch"] = {}
    state["node_links_to_fetch"] = []

    return state


def should_continue_to_router(state: AgentState) -> bool:
    return state["allow_continue"]


def should_continue_to_recursive_retrieval(state: AgentState) -> bool:
    if len(state["node_links_to_fetch"]) + len(state["node_footers_to_fetch"]) > 0:
        return True
    else:
        return False


workflow = StateGraph(AgentState)

workflow.add_node("Initial Search Agent", inital_search_agent)
workflow.add_node("Definition Agent", definition_agent)
workflow.add_node("Context Fetch Tool", context_fetch_tool)
workflow.add_node("Footer Parsing Agent", footer_parsing_agent)
workflow.add_node("Supervisor Agent", supervisor_agent)
workflow.add_node("Router Agent", router_agent)
workflow.add_node("Answering Agent", answering_agent)
workflow.add_node("Recursive Retrieval", recursive_retrieval)

workflow.add_edge(START, "Initial Search Agent")
workflow.add_edge("Initial Search Agent", "Definition Agent")
workflow.add_edge("Definition Agent", "Context Fetch Tool")
workflow.add_edge("Context Fetch Tool", "Footer Parsing Agent")
workflow.add_edge("Footer Parsing Agent", "Supervisor Agent")
# NOTE: the notebook also had an unconditional edge Supervisor -> Router here. Together with the conditional
# edges below it makes Router AND Answering run in parallel whenever the supervisor says END, so it was removed.
workflow.add_conditional_edges(
    "Supervisor Agent",
    should_continue_to_router,
    {
        True: "Router Agent",
        False: "Answering Agent",
    },
)
workflow.add_conditional_edges(
    "Router Agent",
    should_continue_to_recursive_retrieval,
    {
        True: "Recursive Retrieval",
        False: "Answering Agent",
    },
)
workflow.add_edge("Recursive Retrieval", "Context Fetch Tool")
workflow.add_edge("Answering Agent", END)

graph = workflow.compile()


# --------------------------------------------------------------------------------------
# Build everything for one uploaded PDF + Reducto JSON
# --------------------------------------------------------------------------------------
def build_pipeline(
    file_stem: str,
    definitions_page_ranges: List[Tuple[int, int]],
    api_key: str,
    embed_model: str = "text-embedding-3-small",
    progress: Callable[[str], None] = print,
) -> Dict[str, Any]:
    """
    file_stem: path without extension; <file_stem>.pdf and <file_stem>.json must exist.
    definitions_page_ranges: e.g. [(4, 5)] (1-based, inclusive). Empty list = document has no definitions section.
    """
    global FILENAME, LEXICAL_GRAPH, DEFINITIONS_GRAPH, local_nodes_map, vector_index, keyword_index
    global keywords_bm25_retriever, _log

    _log = progress
    get_page_resolution_pymupdf.cache_clear()
    FILENAME = file_stem

    progress("Configuring OpenAI / LlamaIndex...")
    configure_openai(api_key, embed_model)

    # ---- definitions graph
    if definitions_page_ranges:
        progress("Extracting term/definition pairs with GPT-4o...")
        DEFINITIONS_GRAPH = DefinitionGraphBuilder(
            definitions_page_ranges=definitions_page_ranges,
            file_path=f"{file_stem}.pdf",
            definitions_csv_path=os.path.join(GRAPH_DIR, f"{os.path.basename(file_stem)}_definitions.csv"),
            extract_with_llm=True,
        ).build_definitions_graph()
    else:
        DEFINITIONS_GRAPH = nx.DiGraph(name="Legal Definitions Graph")
        progress("No definitions section: definitions graph is empty.")

    # ---- lexical graph
    LEXICAL_GRAPH = NetworkXLexicalGraphBuilder(file_stem).build_graph()
    add_embeddings(LEXICAL_GRAPH, embed_model)
    save_graph(LEXICAL_GRAPH, os.path.join(GRAPH_DIR, f"{os.path.basename(file_stem)}_lexical_graph.pkl"))

    # ---- local nodes
    progress("Creating local nodes + LlamaIndex indices...")
    local_nodes_map = {node_id: graph_node_to_local_node(node_id) for node_id in LEXICAL_GRAPH.nodes}
    lengths = [len(n.embedding) for n in local_nodes_map.values()]
    assert min(lengths) == max(lengths), "All nodes must have the same embedding length"

    storage_context = StorageContext.from_defaults()
    vector_index = VectorStoreIndex(nodes=list(local_nodes_map.values()), storage_context=storage_context)
    keyword_index = SimpleKeywordTableIndex(nodes=list(local_nodes_map.values()), storage_context=storage_context)
    keywords_bm25_retriever = BM25Keyword(keyword_top_k=5, bm25_top_k=3, mode="OR")

    # ---- footer nodes by page
    footer_nodes_by_page_number.clear()
    for node_id, node in local_nodes_map.items():
        if node.type == ElementType.FOOTER:
            footer_nodes_by_page_number[int(node.metadata["Page Number"])].append(node)

    progress("Ready.")
    return {
        "nodes": LEXICAL_GRAPH.number_of_nodes(),
        "edges": LEXICAL_GRAPH.number_of_edges(),
        "nodes_by_type": dict(Counter(d["label"] for _, d in LEXICAL_GRAPH.nodes(data=True))),
        "edges_by_relation": dict(Counter(rel for _, _, rel in LEXICAL_GRAPH.edges(keys=True))),
        "definitions": sum(1 for _, d in DEFINITIONS_GRAPH.nodes(data=True) if d["label"] == "Term"),
        "embed_model": embed_model,
    }


def _flatten_ids(nodes: List[MultiAgentSearchLocalNode]) -> List[str]:
    ids: List[str] = []
    for n in nodes:
        ids.append(n.id_)
        ids.extend(_flatten_ids(list(n.children.values())))
    return ids


def run_query(query: str, on_update: Optional[Callable[[str], None]] = None) -> Dict[str, Any]:
    """Run the multi-agent workflow. on_update(markdown_debug) is called after every agent step."""
    # Retrieved nodes are shared objects and agents write footer/link hints into node.context,
    # so reset them between questions to stop hints from one question leaking into the next.
    for n in local_nodes_map.values():
        n.context = {}
        n.children = {}

    initial_state = AgentState(
        query=query,
        definitions={},
        previous_nodes=[],
        last_fetched_context_nodes=[],
        node_links_to_fetch=[],
        node_footers_to_fetch={},
        allow_continue=True,
        search_failures=[],
        markdown_debug=debug_markdown,
        pass_count=1,
        answer="",
    )

    final_state = initial_state
    for event in graph.stream(initial_state, {"recursion_limit": 150}):
        value = next(iter(event.values()))
        final_state = value
        if on_update is not None:
            try:
                on_update(value["markdown_debug"])
            except KeyError:
                pass

    used_ids = list(dict.fromkeys(_flatten_ids(final_state["previous_nodes"]) + _flatten_ids(final_state["last_fetched_context_nodes"])))
    return {
        "query": query,
        "answer": final_state.get("answer", ""),
        "definitions": final_state.get("definitions", {}),
        "trace": final_state["markdown_debug"],
        "used_node_ids": used_ids,
        "passes": final_state.get("pass_count", 1),
        "search_failures": final_state.get("search_failures", []),
    }


# --------------------------------------------------------------------------------------
# Visualisation (pyvis -> HTML string, rendered by Streamlit)
# --------------------------------------------------------------------------------------
from pyvis.network import Network  # noqa: E402

NODE_COLORS = {
    "Document": "#3ec23e", "Figure": "#f0a45d", "Footer": "#e5173f",
    "Header": "#3cc7d6", "List Item": "#5b6078", "Page": "#8d9be0",
    "Section Header": "#5b2fd0", "Text": "#7a7a7a", "Title": "#0f9fc4", "Table": "#c2a23e",
}
EDGE_COLORS = {"contains": "#9aa0a6", "is_parent": "#4c6ef5", "follows": "#2f9e44", "links_to": "#f08c00"}
DEF_COLORS = {"Term": "#5b2fd0", "Definition": "#3cc7d6"}
DEF_EDGE_COLORS = {"has_definition": "#9aa0a6"}
HIGHLIGHT_BORDER = "#ff1744"

_VIS_OPTIONS = """
var options = {
  "edges": {"font": {"size": 0}, "smooth": false},
  "interaction": {"hover": true, "navigationButtons": true, "multiselect": true, "keyboard": true}
}"""


def lexical_graph_html(G, height="750px", hide_types=(), highlight_ids=(), max_label=22) -> str:
    highlight_ids = set(highlight_ids)
    net = Network(height=height, width="100%", directed=True, notebook=False, cdn_resources="remote",
                  bgcolor="#ffffff", font_color="#333333", filter_menu=True, select_menu=True)

    keep = set()
    for nid, d in G.nodes(data=True):
        etype = d.get("label", "Text")
        if etype in hide_types:
            continue
        keep.add(nid)
        text = (d.get("text") or str(nid)).replace("\n", " ").strip()
        page = (d.get("properties") or {}).get("Page Number", "")
        tooltip = f"{etype}" + (f" (page {page})" if page != "" else "") + f"\n{text[:400]}"
        base = NODE_COLORS.get(etype, "#999999")
        is_hit = nid in highlight_ids
        net.add_node(
            nid,
            label=text[:max_label] + ("…" if len(text) > max_label else ""),
            title=html.escape(tooltip),
            group=etype,
            color={"background": base, "border": HIGHLIGHT_BORDER} if is_hit else base,
            borderWidth=5 if is_hit else 1,
            size=(10 + min(G.degree(nid), 30)) + (12 if is_hit else 0),
        )

    for u, v, rel in G.edges(keys=True):
        if u in keep and v in keep:
            net.add_edge(u, v, title=rel, label=rel, relation=rel,
                         color=EDGE_COLORS.get(rel, "#bbbbbb"), arrows="to", width=1)

    net.barnes_hut(gravity=-6000, central_gravity=0.2, spring_length=120)
    net.set_options(_VIS_OPTIONS)
    return net.generate_html(notebook=False)


def definitions_graph_html(G, height="700px", max_label=28, term_filter=None, highlight_terms=()) -> str:
    highlight_terms = set(highlight_terms)
    net = Network(height=height, width="100%", directed=True, notebook=False, cdn_resources="remote",
                  bgcolor="#ffffff", font_color="#333333", filter_menu=True, select_menu=True)

    keep = set()
    for nid, d in G.nodes(data=True):
        label_type = d.get("label", "Term")
        text = (d.get("text") or str(nid)).replace("\n", " ").strip()

        if term_filter:
            if label_type == "Term" and term_filter.lower() not in text.lower():
                continue
            if label_type == "Definition":
                parent_terms = [u for u, _ in G.in_edges(nid)]
                if not any(term_filter.lower() in (G.nodes[p].get("text") or "").lower() for p in parent_terms):
                    continue
        keep.add(nid)

        is_term = label_type == "Term"
        parents = [u for u, _ in G.in_edges(nid)]
        is_hit = (is_term and nid in highlight_terms) or (not is_term and any(p in highlight_terms for p in parents))
        base = DEF_COLORS.get(label_type, "#999999")
        net.add_node(
            nid,
            label=text[:max_label] + ("…" if len(text) > max_label else ""),
            title=html.escape(f"{label_type}\n{text[:600]}"),
            group=label_type,
            color={"background": base, "border": HIGHLIGHT_BORDER} if is_hit else base,
            borderWidth=5 if is_hit else 1,
            size=(22 if is_term else 12) + (8 if is_hit else 0),
            shape="dot",
        )

    for u, v, d in G.edges(data=True):
        if u in keep and v in keep:
            rel = d.get("relation", "has_definition")
            net.add_edge(u, v, title=rel, label=rel, color=DEF_EDGE_COLORS.get(rel, "#bbbbbb"), arrows="to")

    net.barnes_hut(gravity=-3000, central_gravity=0.3, spring_length=90)
    net.set_options(_VIS_OPTIONS)
    return net.generate_html(notebook=False)


## 2 · Streamlit app (`app.py`)

In [ ]:
%%writefile app.py
import os
import re
from collections import Counter
from pathlib import Path

import fitz
import pandas as pd
import streamlit as st
import streamlit.components.v1 as components

import rag_pipeline as rp

st.set_page_config(page_title="Legal Recursive Retrieval RAG", page_icon="⚖️", layout="wide")
st.title("⚖️ Legal document RAG: recursive retrieval on a lexical graph")


def parse_ranges(text: str, n_pages: int):
    """'4-5, 12' -> [(4, 5), (12, 12)] (1-based, inclusive). Raises ValueError on bad input."""
    ranges = []
    for part in [p.strip() for p in text.split(",") if p.strip()]:
        m = re.fullmatch(r"(\d+)\s*(?:-\s*(\d+))?", part)
        if not m:
            raise ValueError(f"Can't read '{part}'. Use e.g. 4-5, 12")
        start = int(m.group(1))
        end = int(m.group(2) or start)
        if start < 1 or end < start or end > n_pages:
            raise ValueError(f"Range {part} is outside the PDF (1-{n_pages})")
        ranges.append((start, end))
    return ranges


# ----------------------------------------------------------------------------- sidebar
with st.sidebar:
    st.header("1 · Setup")
    api_key = st.text_input("OpenAI API key", type="password", value=os.environ.get("OPENAI_API_KEY", ""))
    embed_model = st.selectbox("Embedding model", ["text-embedding-3-small", "text-embedding-3-large"])
    pdf_file = st.file_uploader("PDF document", type=["pdf"])
    json_file = st.file_uploader("Reducto JSON output of that PDF", type=["json"])

    n_pages = 0
    if pdf_file is not None:
        with fitz.open(stream=pdf_file.getvalue(), filetype="pdf") as _doc:
            n_pages = _doc.page_count
        st.caption(f"PDF has {n_pages} pages")

    has_defs = st.checkbox("Document has a definitions section", value=True)
    def_pages_text = st.text_input("Definition pages (1-based)", value="4-5", disabled=not has_defs,
                                   help="Pages GPT-4o reads to extract term/definition pairs. Ranges like 4-5, 12")

    build_clicked = st.button("Build graphs & index", type="primary", use_container_width=True)

    built = st.session_state.get("built")
    if built:
        st.success("Pipeline ready")
        c1, c2 = st.columns(2)
        c1.metric("Lexical nodes", built["stats"]["nodes"])
        c2.metric("Edges", built["stats"]["edges"])
        st.metric("Defined terms", built["stats"]["definitions"])

# ----------------------------------------------------------------------------- build
if build_clicked:
    if not api_key:
        st.sidebar.error("Enter your OpenAI API key.")
    elif pdf_file is None or json_file is None:
        st.sidebar.error("Upload both the PDF and the Reducto JSON.")
    else:
        try:
            ranges = parse_ranges(def_pages_text, n_pages) if has_defs else []
            pdf_bytes, json_bytes = pdf_file.getvalue(), json_file.getvalue()

            # sanity-check the Reducto JSON structure the pipeline expects
            import json as _json
            data = _json.loads(json_bytes)
            if "result" not in data or "chunks" not in data["result"]:
                raise ValueError("This doesn't look like a Reducto output: expected data['result']['chunks'][...]['blocks'].")

            stem = re.sub(r"[^A-Za-z0-9_-]+", "_", Path(pdf_file.name).stem) or "document"
            Path(f"{stem}.pdf").write_bytes(pdf_bytes)
            Path(f"{stem}.json").write_bytes(json_bytes)

            with st.status("Building graphs and index...", expanded=True) as status:
                stats = rp.build_pipeline(
                    file_stem=stem,
                    definitions_page_ranges=ranges,
                    api_key=api_key,
                    embed_model=embed_model,
                    progress=status.write,
                )
                status.update(label="Pipeline ready", state="complete", expanded=False)
            st.session_state["built"] = {"stem": stem, "stats": stats}
            st.session_state["last_result"] = None
            st.rerun()
        except Exception as e:  # noqa: BLE001
            st.error(f"Build failed: {e}")
            st.exception(e)

# ----------------------------------------------------------------------------- main
if not (st.session_state.get("built") and rp.is_ready()):
    st.info("Upload the PDF and its Reducto JSON in the sidebar, then click **Build graphs & index**.")
    st.stop()

view = st.radio("View", ["Ask a question", "Lexical graph", "Definitions graph"], horizontal=True, label_visibility="collapsed")
last = st.session_state.get("last_result")

# ---- ask
if view == "Ask a question":
    question = st.text_area("Your question", placeholder="e.g. How can the Board and the CCO manage control functions?", height=80)
    ask_clicked = st.button("Ask", type="primary", disabled=not question.strip())

    if ask_clicked:
        # make sure the OpenAI client uses the key currently in the sidebar
        if api_key:
            rp.configure_openai(api_key, st.session_state["built"]["stats"]["embed_model"])
        try:
            with st.status("Running multi-agent retrieval...", expanded=True) as status:
                trace_box = st.empty()
                result = rp.run_query(question.strip(), on_update=lambda md: trace_box.markdown(md))
                status.update(label=f"Done ({result['passes']} pass(es))", state="complete", expanded=False)
            st.session_state["last_result"] = result
            last = result
        except Exception as e:  # noqa: BLE001
            st.error(f"Query failed: {e}")
            st.exception(e)

    if last:
        st.subheader("Answer")
        st.markdown(last["answer"] or "_No answer was produced._")

        st.subheader("Relevant definitions")
        if last["definitions"]:
            for term, definition in last["definitions"].items():
                st.markdown(f"**{term}**: {definition}")
        else:
            st.caption("No defined terms matched this question.")

        if last["search_failures"]:
            with st.expander(f"Search failures ({len(last['search_failures'])})"):
                for f in last["search_failures"]:
                    st.write("- " + f)

        with st.expander(f"Source nodes used ({len(last['used_node_ids'])})"):
            rows = []
            for nid in last["used_node_ids"]:
                d = rp.LEXICAL_GRAPH.nodes[nid]
                rows.append({"type": d["label"], "page": (d.get("properties") or {}).get("Page Number", ""),
                             "text": " ".join((d.get("text") or "").split())[:300]})
            st.dataframe(pd.DataFrame(rows), use_container_width=True, hide_index=True)

        with st.expander("Full agent trace"):
            st.markdown(last["trace"])

# ---- lexical graph
elif view == "Lexical graph":
    G = rp.LEXICAL_GRAPH
    stats = st.session_state["built"]["stats"]
    st.caption("Nodes by type: " + ", ".join(f"{k}: {v}" for k, v in stats["nodes_by_type"].items()))
    st.caption("Edges by relation: " + ", ".join(f"{k}: {v}" for k, v in stats["edges_by_relation"].items()))

    c1, c2, c3 = st.columns([2, 2, 1])
    types_present = sorted(stats["nodes_by_type"].keys())
    hide_types = c1.multiselect("Hide element types", types_present, help="Hiding Footer / Page / Figure makes big graphs easier to read")
    do_highlight = c2.checkbox("Highlight nodes used in the last answer", value=bool(last), disabled=not last)
    height = c3.slider("Height (px)", 500, 1200, 750, step=50)

    with st.spinner("Rendering graph..."):
        page = rp.lexical_graph_html(
            G, height=f"{height}px", hide_types=tuple(hide_types),
            highlight_ids=(last["used_node_ids"] if (last and do_highlight) else ()),
        )
    components.html(page, height=height + 160, scrolling=True)

    legend = " ".join(
        f"<span style='display:inline-block;margin-right:12px'><span style='color:{c}'>●</span> {t}</span>"
        for t, c in rp.NODE_COLORS.items() if t in types_present
    )
    edge_legend = " ".join(
        f"<span style='display:inline-block;margin-right:12px'><span style='color:{c}'>━</span> {r}</span>"
        for r, c in rp.EDGE_COLORS.items()
    )
    st.markdown(legend + "<br>" + edge_legend, unsafe_allow_html=True)

# ---- definitions graph
else:
    G = rp.DEFINITIONS_GRAPH
    terms = [(n, d["text"]) for n, d in G.nodes(data=True) if d["label"] == "Term"]
    if not terms:
        st.info("This document has no definitions graph (no definition pages were given).")
        st.stop()

    c1, c2, c3 = st.columns([2, 2, 1])
    term_filter = c1.text_input("Filter terms (substring)", "")
    do_highlight = c2.checkbox("Highlight terms matched by the last question", value=bool(last and last["definitions"]),
                               disabled=not (last and last["definitions"]))
    height = c3.slider("Height (px)", 400, 1100, 650, step=50)

    page = rp.definitions_graph_html(
        G, height=f"{height}px", term_filter=term_filter or None,
        highlight_terms=(last["definitions"].keys() if (last and do_highlight) else ()),
    )
    components.html(page, height=height + 160, scrolling=True)

    with st.expander(f"All defined terms ({len(terms)})"):
        defs = {n: next((G.nodes[s]["text"] for s in G.successors(n)), "") for n, _ in terms}
        st.dataframe(pd.DataFrame({"Term": list(defs.keys()), "Definition": list(defs.values())}),
                     use_container_width=True, hide_index=True)


## 3 · Launch the app
Starts Streamlit and exposes it through a free Cloudflare quick tunnel. Re-run this cell to restart.

In [ ]:
import os, re, subprocess, time

# stop anything left over from a previous run
subprocess.run("pkill -f 'streamlit run' ; pkill -f cloudflared", shell=True)
time.sleep(1)

# cloudflared (one-time download)
if not os.path.exists("cloudflared"):
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared", shell=True, check=True)

# Streamlit (XSRF/CORS protection must be off for file uploads to work through a tunnel)
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
     "--browser.gatherUsageStats", "false"],
    stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT,
)
time.sleep(5)

# tunnel
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                 stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0)
        break
print("Open the app here:", url or "tunnel URL not found, check tunnel.log")

## Troubleshooting
- Blank page or error right after opening the URL: wait ~10 s and refresh, or run `!tail -n 30 streamlit.log`.
- No URL printed: run `!cat tunnel.log`.
- Uploaded files and caches live in the Colab working directory (`<name>.pdf`, `<name>.json`, `graph_data/`, `docs_links/`) and vanish when the runtime resets.

## Differences from the original notebook
- **Same logic**: link detection, definitions graph, lexical graph, hybrid retrievers, pruning, clause expansion, footer/link hints, supervisor/router/answering agents and the LangGraph workflow are copied over; the notebook's module-level globals are filled in by `build_pipeline()`.
- Removed the unconditional `Supervisor → Router` edge. Combined with the conditional edges, it ran Router and Answering in parallel whenever the supervisor returned END, which LangGraph rejects because both write the same state keys.
- The over-token-limit branch in the supervisor called `count_tokens()` on a list; it now uses `count_tokens_for_nodes()`.
- Node `context` is reset at the start of each question, so link/footer hints from one question don't leak into the next (the notebook only ever ran one).
- `definitions_search` returns `{}` without an LLM call when the document has no definitions section.
- PDF page sizes are cached (`lru_cache`) instead of reopening the PDF for every element/link comparison.
- The final answer is stored in the state (`answer`) so the app can display it separately from the trace.